<a href="https://colab.research.google.com/github/satyammishra4049-eng/AI-skillmap-Agent/blob/main/Skillmap_Career_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U langchain-google-genai
!pip install langchain langchain-tavily
!pip install langchain-mcp-adapters

In [ ]:
from google.colab import userdata

google_api_key = userdata.get('GEMINI_API_KEY')
tavily_api_key = userdata.get('TAVILY_API_KEY')
rapidapi_key = userdata.get('rapidapi_key')

In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model(
    "google_genai:gemini-2.5-flash",
    api_key=google_api_key
)

In [ ]:
from langchain_tavily import TavilySearch

skill_demand_tool = TavilySearch(
    max_results=5,
    search_depth="advanced",
    tavily_api_key=tavily_api_key,
)

In [ ]:
import requests
from langchain.tools import tool

@tool
def search_jobs(skill: str, location: str) -> list:
    """Search jobs based on skill and location using RapidAPI"""

    url = "https://jsearch.p.rapidapi.com/search"

    headers = {
        "x-rapidapi-key": rapidapi_key,
        "x-rapidapi-host": "jsearch.p.rapidapi.com"
    }

    querystring = {
        "query": f"{skill} in {location}",
        "page": "1",
        "country": "in"
    }

    response = requests.get(url, headers=headers, params=querystring)
    data = response.json()

    jobs = data.get("data", [])

    result = []

    for job in jobs:
        result.append({
            "title": job.get("job_title"),
            "company": job.get("employer_name"),
            "location": job.get("job_city"),
            "apply_link": job.get("job_apply_link")
        })

    return result

In [ ]:
system_prompt = """You are a SkillMap Career Agent.

You help students:
1. Understand skill demand
2. Find job opportunities

Always give:
- Skill demand
- Career scope
- Job listings with links

Keep response clean and readable.
"""

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "mcp_tavily": {
            "transport": "http",
            "url": "https://backend.composio.dev"
        }
    }
)

In [ ]:
from langchain.agents import create_agent
import asyncio

async def run_agent():

    user_query = input("Enter your skill: ")
    location = input("Enter location (e.g., Pune, Mumbai): ")

    all_tools = [search_jobs]

    agent = create_agent(
        model=model,
        tools=all_tools,
        system_prompt=system_prompt
    )

    response = await agent.ainvoke({
        "messages": [{"role": "user", "content": f"{user_query} jobs in {location}"}]
    })

    print("\n===== RESULT =====\n")


    print(response["messages"][-1].content)

await run_agent()

# SkillMap Career Agent - Project Explanation

1. Project Overview  
This project is an AI-powered SkillMap Career Agent that helps students understand the demand of a particular skill and find relevant job opportunities based on that skill and location.

2. Objective  
The main objective of this project is to bridge the gap between student skills and industry requirements by providing real-time career insights and job listings.

3. Technologies Used  
- Google Gemini (LLM) for intelligent response generation  
- LangChain for building AI agent and tool integration  
- RapidAPI (JSearch API) for fetching real-time job listings  
- Python for implementation  
- Google Colab for development and execution  

4. Working Process  
- User enters a skill (e.g., Python, AI, Web Development)  
- User enters a location (e.g., Pune, Mumbai)  
- The AI agent processes the query  
- The system uses the job search tool to fetch real job data  
- The output is displayed with job title, company name, location, and apply link  

5. Features  
- Skill-based career guidance  
- Real-time job search  
- AI-powered intelligent responses  
- Easy-to-use interface  

6. Real-world Application  
This system can be used by students, freshers, and job seekers to explore career opportunities and understand industry demand for different skills.

7. Conclusion  
The SkillMap Career Agent simplifies career decision-making by combining AI intelligence with real-time job data, making it a useful tool for modern students.